## Portable path configuration
This release copy contains no personal or NAS paths. Set `GAO2026_DATA_ROOT` for deposited data and `GAO2026_RAW_ROOT` only when running raw-image stages (raw revision microscopy is not included).


In [ ]:
from pathlib import Path
import os
_default_repo_root = Path.cwd().resolve()
GAO2026_DATA_ROOT = Path(os.environ.get('GAO2026_DATA_ROOT', _default_repo_root / 'data')).expanduser().resolve()
GAO2026_RAW_ROOT = Path(os.environ.get('GAO2026_RAW_ROOT', GAO2026_DATA_ROOT / 'external_raw')).expanduser().resolve()
GAO2026_OUTPUT_ROOT = Path(os.environ.get('GAO2026_OUTPUT_ROOT', _default_repo_root / 'outputs')).expanduser().resolve()
GAO2026_EXTERNAL_ROOT = Path(os.environ.get('GAO2026_EXTERNAL_ROOT', GAO2026_DATA_ROOT / 'external')).expanduser().resolve()


# 200-nm bead comparison: conventional CSU-W1 versus SoRa

**Objective.** Quantify the observed image width of the same 200-nm fluorescent
beads acquired sequentially with conventional CSU-W1 and SoRa, and generate the
source data and publication-ready Supplementary Figure used to document the
effective lateral imaging improvement.

**Interpretation boundary.** These measurements contain the finite 200-nm bead
diameter and come from one technical acquisition per modality. They are therefore
reported as *observed bead image widths*, not absolute point-source resolution or
biological-replicate statistics.


## Analysis plan

1. Read acquisition metadata directly from both ND2 files and verify the objective,
   numerical aperture, Z step, image dimensions, and SoRa flag.
2. Refit both raw stacks with the same candidate-detection and anisotropic 3D
   Gaussian-fitting implementation. The primary comparison uses unfiltered raw
   images for conventional CSU-W1 and the 3x3 XY median filtering used for the
   fixed-cell SoRa analysis for SoRa. All-raw fits are retained as a sensitivity
   analysis.
3. Use the known 2.8x sampling ratio and a robust translation estimate to identify
   corresponding beads in the overlapping physical field. Accept one-to-one pairs
   with residual XY distance no greater than 200 nm. Exclude matched aggregates when
   both acquisitions exceed their channel-specific upper Tukey FWHM fence.
4. Define observed lateral FWHM as the geometric mean of the fitted X and Y FWHM.
   Summarize paired changes without treating individual beads as independent
   experimental replicates. Bootstrap intervals describe within-field precision.
5. Export a publication figure, source-data tables, metadata, a methods/legend draft,
   and machine-readable validation results.


In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import math
import os
import subprocess
import sys
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import nd2
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from scipy.ndimage import median_filter
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree

SEED = 20260810
RNG = np.random.default_rng(SEED)

NORMAL_BEAD_ND2 = Path(
    str(GAO2026_RAW_ROOT / 'Public-TS-873-2/Microscope/Ko/260716-SoRa_Fixed/260716_SoRa/260716_beads/200nm_CYR_001.nd2')
)
SORA_BEAD_ND2 = Path(
    str(GAO2026_RAW_ROOT / 'Public-TS-873-2/Microscope/Ko/260716-SoRa_Fixed/260716_SoRa/260716_beads/200nm_CYR_SoRa_001.nd2')
)
NORMAL_SAMPLE_ND2 = Path(
    str(GAO2026_RAW_ROOT / 'Public-TS-873-2/Microscope/Ko/260716-SoRa_Fixed/260716_SoRa/260716_ChamberA/ChamberA_20260716_205221_Field4.nd2')
)
SORA_SAMPLE_ND2 = Path(
    str(GAO2026_RAW_ROOT / 'Public-TS-873-2/Microscope/Ko/260716-SoRa_Fixed/260716_SoRa/260716_ChamberA/ChamberA_20260716_205221_Field1.nd2')
)

DEFAULT_OUTPUT_ROOT = Path(
    str(GAO2026_RAW_ROOT / 'Public-TS-873-2/Microscope/Ko/260717-SoRa_Fixed/ 260718-analysis-HO/bead_normal_vs_sora_publication_20260810')
)
OUTPUT_ROOT = Path("outputs")
FIT_ROOT = OUTPUT_ROOT / "fit_results"
FIGURE_DIR = OUTPUT_ROOT / "figures"
TABLE_DIR = OUTPUT_ROOT / "source_data"
REPORT_DIR = OUTPUT_ROOT / "reports"
for directory in (FIT_ROOT, FIGURE_DIR, TABLE_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

notebook_dir_candidates = [
    Path.cwd(),
    Path.cwd().parent / "scripts",
    Path.cwd() / "scripts",
    Path.cwd() / "bead_publication_notebook_20260810",
    OUTPUT_ROOT,
]
PIPELINE_SCRIPT = next(
    (directory / "bead_fit_pipeline.py" for directory in notebook_dir_candidates
     if (directory / "bead_fit_pipeline.py").exists()),
    None,
)
if PIPELINE_SCRIPT is None:
    raise FileNotFoundError(
        "bead_fit_pipeline.py was not found. Run the notebook from its directory "
        "or set BEAD_OUTPUT_ROOT to a directory containing the sidecar module."
    )

PRIMARY_PREPROCESSING_BY_MODALITY = {
    "Conventional CSU-W1": "raw",
    "SoRa": "median3x3",
}
MIN_FIT_R_SQUARED = 0.80
MAX_PAIR_DISTANCE_NM = 200.0
COARSE_PAIR_DISTANCE_NM = 1500.0
MAX_CANDIDATES = 120
FORCE_REFIT = os.environ.get("BEAD_FORCE_REFIT", "0") == "1"
N_BOOTSTRAP = 10_000

CHANNEL_LABELS = {0: "640 nm", 1: "515 nm", 2: "445 nm"}
CHANNEL_COLORS = {0: "#B83B3B", 1: "#2C8C5A", 2: "#2C78B8"}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.5,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

print(f"Python: {sys.executable}")
print(f"Pipeline: {PIPELINE_SCRIPT}")
print(f"Output: {OUTPUT_ROOT}")


## 1. Acquisition metadata and preflight checks


In [ ]:
spec = importlib.util.spec_from_file_location("bead_fit_pipeline", PIPELINE_SCRIPT)
bead_fit_pipeline = importlib.util.module_from_spec(spec)
assert spec.loader is not None
sys.modules[spec.name] = bead_fit_pipeline
spec.loader.exec_module(bead_fit_pipeline)

normal_metadata = bead_fit_pipeline.read_optical_metadata(NORMAL_BEAD_ND2)
sora_metadata = bead_fit_pipeline.read_optical_metadata(SORA_BEAD_ND2)

def nd2_shape(path: Path) -> tuple[int, int, int, int]:
    with nd2.ND2File(path) as image:
        return tuple(int(image.sizes[axis]) for axis in ("Z", "C", "Y", "X"))

normal_shape = nd2_shape(NORMAL_BEAD_ND2)
sora_shape = nd2_shape(SORA_BEAD_ND2)

metadata_table = pd.DataFrame([
    {
        "modality": "Conventional CSU-W1",
        "objective": normal_metadata.objective_name,
        "magnification": normal_metadata.objective_magnification,
        "NA": normal_metadata.objective_na,
        "zoom": normal_metadata.zoom_magnification,
        "SoRa flag": normal_metadata.sora,
        "Z/C/Y/X": str(normal_shape),
        "XY sampling (nm/px)": normal_metadata.voxel_x_nm,
        "Z step (nm)": normal_metadata.voxel_z_nm,
    },
    {
        "modality": "SoRa",
        "objective": sora_metadata.objective_name,
        "magnification": sora_metadata.objective_magnification,
        "NA": sora_metadata.objective_na,
        "zoom": sora_metadata.zoom_magnification,
        "SoRa flag": sora_metadata.sora,
        "Z/C/Y/X": str(sora_shape),
        "XY sampling (nm/px)": sora_metadata.voxel_x_nm,
        "Z step (nm)": sora_metadata.voxel_z_nm,
    },
])

assert normal_metadata.objective_name == sora_metadata.objective_name
assert math.isclose(normal_metadata.objective_na, sora_metadata.objective_na)
assert math.isclose(normal_metadata.objective_magnification, sora_metadata.objective_magnification)
assert math.isclose(normal_metadata.voxel_z_nm, sora_metadata.voxel_z_nm)
assert normal_shape == sora_shape
assert not normal_metadata.sora and sora_metadata.sora
assert len(normal_metadata.channel_names) == len(sora_metadata.channel_names) == 3

sampling_ratio = normal_metadata.voxel_x_nm / sora_metadata.voxel_x_nm
assert math.isclose(sampling_ratio, 2.8, rel_tol=1e-6)

metadata_table.to_csv(TABLE_DIR / "acquisition_metadata.csv", index=False)
display(metadata_table)


## 2. Raw ND2 fitting


In [ ]:
def run_fit(modality: str, bead_path: Path, sample_path: Path) -> Path:
    output_dir = FIT_ROOT / modality.lower().replace(" ", "_")
    fit_csv = output_dir / "bead_gaussian_fits.csv"
    required = [
        fit_csv,
        output_dir / "optical_metadata.json",
        output_dir / "parameter_recommendations.json",
    ]
    if not FORCE_REFIT and all(path.exists() for path in required):
        print(f"Reuse {modality}: {fit_csv}")
        return fit_csv

    command = [
        sys.executable,
        str(PIPELINE_SCRIPT),
        "--bead", str(bead_path),
        "--sample", str(sample_path),
        "--output", str(output_dir),
        "--max-candidates", str(MAX_CANDIDATES),
    ]
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)
    return fit_csv


normal_fit_csv = run_fit(
    "Conventional CSU-W1", NORMAL_BEAD_ND2, NORMAL_SAMPLE_ND2
)
sora_fit_csv = run_fit("SoRa", SORA_BEAD_ND2, SORA_SAMPLE_ND2)

normal_fits = pd.read_csv(normal_fit_csv).assign(modality="Conventional CSU-W1")
sora_fits = pd.read_csv(sora_fit_csv).assign(modality="SoRa")
all_fits = pd.concat([normal_fits, sora_fits], ignore_index=True)
all_fits["channel_label"] = all_fits["channel_index"].map(CHANNEL_LABELS)

def accepted_mask(frame: pd.DataFrame, preprocessing: str) -> pd.Series:
    return (
        frame["preprocessing"].eq(preprocessing)
        & frame["r_squared"].ge(MIN_FIT_R_SQUARED)
        & frame["sigma_xy_px"].between(0.7, 10.0)
        & frame["sigma_z_px"].between(0.3, 5.0)
    )

primary_fit_frames = []
accepted_count_frames = []
for modality, preprocessing in PRIMARY_PREPROCESSING_BY_MODALITY.items():
    modality_fits = all_fits.loc[all_fits["modality"].eq(modality)].copy()
    modality_fits["accepted"] = accepted_mask(modality_fits, preprocessing)

    primary_fit_frames.append(
        modality_fits.loc[
            modality_fits["preprocessing"].eq(preprocessing)
            & modality_fits["accepted"]
        ].copy()
    )
    accepted_count_frames.append(
        modality_fits.loc[modality_fits["preprocessing"].eq(preprocessing)]
        .assign(primary_preprocessing=preprocessing)
        .groupby(
            [
                "modality",
                "primary_preprocessing",
                "channel_index",
                "channel_label",
            ],
            as_index=False,
        )
        .agg(candidates=("candidate_index", "size"), accepted=("accepted", "sum"))
    )

primary_fits = pd.concat(primary_fit_frames, ignore_index=True)
assert set(
    primary_fits.loc[
        primary_fits["modality"].eq("Conventional CSU-W1"), "preprocessing"
    ]
) == {"raw"}
assert set(
    primary_fits.loc[primary_fits["modality"].eq("SoRa"), "preprocessing"]
) == {"median3x3"}
primary_fits.to_csv(TABLE_DIR / "all_accepted_primary_fits.csv", index=False)

accepted_counts = pd.concat(accepted_count_frames, ignore_index=True)
accepted_counts.to_csv(TABLE_DIR / "accepted_fit_counts.csv", index=False)
display(accepted_counts)


## 3. Register the overlapping field, match beads, and exclude aggregates

Matched pairs are excluded as aggregates only when both conventional CSU-W1 and
SoRa lateral FWHM exceed the channel-specific upper Tukey fence (Q3 + 1.5 x IQR).
The complete pre-QC table and excluded rows are retained as source data.


In [ ]:
def pair_channel(channel_index: int) -> tuple[pd.DataFrame, dict[str, float]]:
    normal = (
        primary_fits.query(
            "modality == 'Conventional CSU-W1' and channel_index == @channel_index"
        )
        .sort_values("candidate_index")
        .reset_index(drop=True)
    )
    sora = (
        primary_fits.query("modality == 'SoRa' and channel_index == @channel_index")
        .sort_values("candidate_index")
        .reset_index(drop=True)
    )

    normal_center_yx = np.array([(normal_shape[2] - 1) / 2, (normal_shape[3] - 1) / 2])
    sora_center_yx = np.array([(sora_shape[2] - 1) / 2, (sora_shape[3] - 1) / 2])
    predicted_sora_yx = (
        (normal[["y", "x"]].to_numpy(float) - normal_center_yx) * sampling_ratio
        + sora_center_yx
    )
    inside = (
        (predicted_sora_yx[:, 0] >= 0)
        & (predicted_sora_yx[:, 0] < sora_shape[2])
        & (predicted_sora_yx[:, 1] >= 0)
        & (predicted_sora_yx[:, 1] < sora_shape[3])
    )
    normal = normal.loc[inside].reset_index(drop=True)
    predicted_sora_yx = predicted_sora_yx[inside]
    sora_yx = sora[["y", "x"]].to_numpy(float)

    tree = cKDTree(sora_yx)
    coarse_distance_px = COARSE_PAIR_DISTANCE_NM / sora_metadata.voxel_x_nm
    pair_distance_px = MAX_PAIR_DISTANCE_NM / sora_metadata.voxel_x_nm

    distances, indices = tree.query(predicted_sora_yx, k=1)
    coarse = distances <= coarse_distance_px
    if coarse.sum() < 5:
        raise RuntimeError(f"Too few coarse matches for channel {channel_index}")
    translation_yx_px = np.median(
        sora_yx[indices[coarse]] - predicted_sora_yx[coarse], axis=0
    )

    for _ in range(3):
        distances, indices = tree.query(predicted_sora_yx + translation_yx_px, k=1)
        close = distances <= pair_distance_px
        if close.sum() < 5:
            raise RuntimeError(f"Too few refined matches for channel {channel_index}")
        translation_yx_px = np.median(
            sora_yx[indices[close]] - predicted_sora_yx[close], axis=0
        )

    registered_yx = predicted_sora_yx + translation_yx_px
    cost_px = np.sqrt(
        ((registered_yx[:, None, :] - sora_yx[None, :, :]) ** 2).sum(axis=2)
    )
    normal_indices, sora_indices = linear_sum_assignment(cost_px)
    keep = cost_px[normal_indices, sora_indices] <= pair_distance_px
    normal_indices = normal_indices[keep]
    sora_indices = sora_indices[keep]

    rows = []
    for normal_index, sora_index in zip(normal_indices, sora_indices):
        n = normal.iloc[normal_index]
        s = sora.iloc[sora_index]
        rows.append({
            "channel_index": channel_index,
            "channel_label": CHANNEL_LABELS[channel_index],
            "normal_candidate_index": int(n["candidate_index"]),
            "sora_candidate_index": int(s["candidate_index"]),
            "normal_preprocessing": str(n["preprocessing"]),
            "sora_preprocessing": str(s["preprocessing"]),
            "normal_z": int(n["z"]),
            "normal_y": int(n["y"]),
            "normal_x": int(n["x"]),
            "sora_z": int(s["z"]),
            "sora_y": int(s["y"]),
            "sora_x": int(s["x"]),
            "pair_distance_nm": float(cost_px[normal_index, sora_index] * sora_metadata.voxel_x_nm),
            "normal_fwhm_xy_nm": float(n["fwhm_xy_nm"]),
            "sora_fwhm_xy_nm": float(s["fwhm_xy_nm"]),
            "normal_fwhm_z_nm": float(n["fwhm_z_nm"]),
            "sora_fwhm_z_nm": float(s["fwhm_z_nm"]),
            "normal_r_squared": float(n["r_squared"]),
            "sora_r_squared": float(s["r_squared"]),
        })

    pairs = pd.DataFrame(rows)
    pairs["paired_reduction_percent"] = 100.0 * (
        1.0 - pairs["sora_fwhm_xy_nm"] / pairs["normal_fwhm_xy_nm"]
    )
    pairs["z_index_difference_sora_minus_normal"] = pairs["sora_z"] - pairs["normal_z"]

    registration = {
        "channel_index": channel_index,
        "channel_label": CHANNEL_LABELS[channel_index],
        "sampling_ratio_normal_to_sora": float(sampling_ratio),
        "translation_y_px_sora_grid": float(translation_yx_px[0]),
        "translation_x_px_sora_grid": float(translation_yx_px[1]),
        "translation_y_nm": float(translation_yx_px[0] * sora_metadata.voxel_y_nm),
        "translation_x_nm": float(translation_yx_px[1] * sora_metadata.voxel_x_nm),
        "normal_accepted_inside_sora_fov": int(len(normal)),
        "sora_accepted": int(len(sora)),
        "paired_beads": int(len(pairs)),
        "maximum_pair_residual_nm": float(pairs["pair_distance_nm"].max()),
        "median_z_index_difference": float(
            pairs["z_index_difference_sora_minus_normal"].median()
        ),
    }
    return pairs, registration


pair_frames = []
registration_rows = []
for channel_index in CHANNEL_LABELS:
    pairs, registration = pair_channel(channel_index)
    pair_frames.append(pairs)
    registration_rows.append(registration)

paired_beads_before_aggregate_qc = pd.concat(pair_frames, ignore_index=True)


def apply_aggregate_qc(frame: pd.DataFrame) -> pd.DataFrame:
    qc_frames = []
    for _, channel_pairs in frame.groupby("channel_index", sort=True):
        channel_pairs = channel_pairs.copy()
        normal_q1, normal_q3 = channel_pairs["normal_fwhm_xy_nm"].quantile(
            [0.25, 0.75]
        )
        sora_q1, sora_q3 = channel_pairs["sora_fwhm_xy_nm"].quantile(
            [0.25, 0.75]
        )
        normal_upper_fence = float(normal_q3 + 1.5 * (normal_q3 - normal_q1))
        sora_upper_fence = float(sora_q3 + 1.5 * (sora_q3 - sora_q1))

        channel_pairs["normal_upper_tukey_fence_nm"] = normal_upper_fence
        channel_pairs["sora_upper_tukey_fence_nm"] = sora_upper_fence
        channel_pairs["normal_upper_fwhm_outlier"] = channel_pairs[
            "normal_fwhm_xy_nm"
        ].gt(normal_upper_fence)
        channel_pairs["sora_upper_fwhm_outlier"] = channel_pairs[
            "sora_fwhm_xy_nm"
        ].gt(sora_upper_fence)
        channel_pairs["excluded_as_aggregate"] = (
            channel_pairs["normal_upper_fwhm_outlier"]
            & channel_pairs["sora_upper_fwhm_outlier"]
        )
        qc_frames.append(channel_pairs)

    return pd.concat(qc_frames, ignore_index=True)


paired_bead_aggregate_qc = apply_aggregate_qc(paired_beads_before_aggregate_qc)
excluded_aggregates = paired_bead_aggregate_qc.loc[
    paired_bead_aggregate_qc["excluded_as_aggregate"]
].copy()
paired_beads = paired_bead_aggregate_qc.loc[
    ~paired_bead_aggregate_qc["excluded_as_aggregate"]
].copy()

aggregate_qc_summary = (
    paired_bead_aggregate_qc.groupby(
        ["channel_index", "channel_label"], as_index=False
    )
    .agg(
        matched_before_qc=("excluded_as_aggregate", "size"),
        excluded_aggregates=("excluded_as_aggregate", "sum"),
    )
)
aggregate_qc_summary["retained_after_qc"] = (
    aggregate_qc_summary["matched_before_qc"]
    - aggregate_qc_summary["excluded_aggregates"]
)

registration_summary = pd.DataFrame(registration_rows).merge(
    aggregate_qc_summary,
    on=["channel_index", "channel_label"],
    how="left",
    validate="one_to_one",
)

paired_bead_aggregate_qc.to_csv(
    TABLE_DIR / "paired_bead_aggregate_qc.csv", index=False
)
excluded_aggregates.to_csv(
    TABLE_DIR / "excluded_bead_aggregates.csv", index=False
)
paired_beads.to_csv(TABLE_DIR / "paired_bead_source_data.csv", index=False)
aggregate_qc_summary.to_csv(
    TABLE_DIR / "aggregate_qc_summary.csv", index=False
)
registration_summary.to_csv(TABLE_DIR / "registration_summary.csv", index=False)

display(registration_summary)
display(aggregate_qc_summary)
display(excluded_aggregates)
display(paired_beads.head())


## 4. Paired summary and raw-image sensitivity


In [ ]:
def bootstrap_median_interval(values: np.ndarray, n_bootstrap: int = N_BOOTSTRAP) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    indices = RNG.integers(0, len(values), size=(n_bootstrap, len(values)))
    bootstrapped = np.median(values[indices], axis=1)
    low, high = np.quantile(bootstrapped, [0.025, 0.975])
    return float(low), float(high)


summary_rows = []
for channel_index, pairs in paired_beads.groupby("channel_index", sort=True):
    reductions = pairs["paired_reduction_percent"].to_numpy(float)
    ci_low, ci_high = bootstrap_median_interval(reductions)
    summary_rows.append({
        "channel_index": int(channel_index),
        "channel_label": CHANNEL_LABELS[int(channel_index)],
        "paired_beads": int(len(pairs)),
        "normal_median_fwhm_xy_nm": float(pairs["normal_fwhm_xy_nm"].median()),
        "normal_q1_fwhm_xy_nm": float(pairs["normal_fwhm_xy_nm"].quantile(0.25)),
        "normal_q3_fwhm_xy_nm": float(pairs["normal_fwhm_xy_nm"].quantile(0.75)),
        "sora_median_fwhm_xy_nm": float(pairs["sora_fwhm_xy_nm"].median()),
        "sora_q1_fwhm_xy_nm": float(pairs["sora_fwhm_xy_nm"].quantile(0.25)),
        "sora_q3_fwhm_xy_nm": float(pairs["sora_fwhm_xy_nm"].quantile(0.75)),
        "median_paired_reduction_percent": float(np.median(reductions)),
        "bootstrap_95ci_low_percent": ci_low,
        "bootstrap_95ci_high_percent": ci_high,
        "fraction_sora_narrower": float(np.mean(reductions > 0)),
    })

paired_summary = pd.DataFrame(summary_rows)

# The all-raw sensitivity analysis uses the same matched candidate identities and
# additionally requires acceptable raw SoRa fits. The 445-nm SoRa raw fits are expected
# to fail the prespecified R2 threshold and are reported as unavailable.
raw_lookup = {
    (row.modality, int(row.channel_index), int(row.candidate_index)): row
    for row in all_fits.loc[accepted_mask(all_fits, "raw")].itertuples(index=False)
}
raw_rows = []
for pair in paired_beads.itertuples(index=False):
    normal_key = ("Conventional CSU-W1", int(pair.channel_index), int(pair.normal_candidate_index))
    sora_key = ("SoRa", int(pair.channel_index), int(pair.sora_candidate_index))
    if normal_key not in raw_lookup or sora_key not in raw_lookup:
        continue
    normal_row = raw_lookup[normal_key]
    sora_row = raw_lookup[sora_key]
    raw_rows.append({
        "channel_index": int(pair.channel_index),
        "channel_label": pair.channel_label,
        "normal_candidate_index": int(pair.normal_candidate_index),
        "sora_candidate_index": int(pair.sora_candidate_index),
        "normal_raw_fwhm_xy_nm": float(normal_row.fwhm_xy_nm),
        "sora_raw_fwhm_xy_nm": float(sora_row.fwhm_xy_nm),
        "raw_paired_reduction_percent": 100.0 * (
            1.0 - float(sora_row.fwhm_xy_nm) / float(normal_row.fwhm_xy_nm)
        ),
    })
raw_paired = pd.DataFrame(raw_rows)

raw_summary_rows = []
for channel_index in CHANNEL_LABELS:
    subset = raw_paired.query("channel_index == @channel_index") if not raw_paired.empty else pd.DataFrame()
    raw_summary_rows.append({
        "channel_index": channel_index,
        "channel_label": CHANNEL_LABELS[channel_index],
        "raw_paired_beads": int(len(subset)),
        "raw_normal_median_fwhm_xy_nm": (
            float(subset["normal_raw_fwhm_xy_nm"].median()) if len(subset) else np.nan
        ),
        "raw_sora_median_fwhm_xy_nm": (
            float(subset["sora_raw_fwhm_xy_nm"].median()) if len(subset) else np.nan
        ),
        "raw_median_paired_reduction_percent": (
            float(subset["raw_paired_reduction_percent"].median()) if len(subset) else np.nan
        ),
    })
raw_summary = pd.DataFrame(raw_summary_rows)

paired_summary.to_csv(TABLE_DIR / "paired_bead_summary.csv", index=False)
raw_paired.to_csv(TABLE_DIR / "paired_bead_raw_sensitivity.csv", index=False)
raw_summary.to_csv(TABLE_DIR / "paired_bead_raw_sensitivity_summary.csv", index=False)
display(paired_summary.round(2))
display(raw_summary.round(2))


## 5. Publication figure


In [ ]:
def normalized_local_projection(
    nd2_path: Path,
    channel_index: int,
    z: int,
    y: int,
    x: int,
    xy_radius_px: int,
    preprocessing: str,
    z_radius: int = 1,
) -> np.ndarray:
    with nd2.ND2File(nd2_path) as image:
        z_size = int(image.sizes["Z"])
        y_size = int(image.sizes["Y"])
        x_size = int(image.sizes["X"])
        z0, z1 = max(0, z - z_radius), min(z_size, z + z_radius + 1)
        y0, y1 = max(0, y - xy_radius_px), min(y_size, y + xy_radius_px + 1)
        x0, x1 = max(0, x - xy_radius_px), min(x_size, x + xy_radius_px + 1)
        local_stack = np.asarray(
            image.to_dask()[z0:z1, channel_index, y0:y1, x0:x1].compute()
        )
    if preprocessing == "median3x3":
        local_stack = median_filter(local_stack, size=(1, 3, 3))
    elif preprocessing != "raw":
        raise ValueError(f"Unsupported preprocessing: {preprocessing}")
    crop = local_stack.max(axis=0).astype(np.float32)
    background = float(np.percentile(crop, 10.0))
    crop = np.clip(crop - background, 0.0, None)
    scale = float(np.percentile(crop, 99.8))
    return np.clip(crop / scale, 0.0, 1.0) if scale > 0 else crop


# Select the retained pair closest to the channel-wise bivariate median of the
# conventional and SoRa absolute FWHM values. Fit quality and registration residual
# provide deterministic tie-breakers.
representative_pairs = {}
for channel_index, pairs in paired_beads.groupby("channel_index", sort=True):
    normal_median = float(pairs["normal_fwhm_xy_nm"].median())
    sora_median = float(pairs["sora_fwhm_xy_nm"].median())
    candidates = pairs.assign(
        distance_to_bivariate_median=np.hypot(
            np.log(pairs["normal_fwhm_xy_nm"] / normal_median),
            np.log(pairs["sora_fwhm_xy_nm"] / sora_median),
        ),
        minimum_r_squared=pairs[["normal_r_squared", "sora_r_squared"]].min(axis=1),
    ).sort_values(
        ["distance_to_bivariate_median", "minimum_r_squared", "pair_distance_nm"],
        ascending=[True, False, True],
    )
    representative_pairs[int(channel_index)] = candidates.iloc[0]

figure = plt.figure(figsize=(11.0, 7.2), constrained_layout=False)
outer = figure.add_gridspec(2, 2, height_ratios=[1.05, 1.0], width_ratios=[1.35, 0.8], hspace=0.42, wspace=0.34)
image_grid = outer[0, :].subgridspec(2, 3, hspace=0.12, wspace=0.12)

crop_radius_um = 0.75
normal_radius_px = int(round(crop_radius_um * 1000.0 / normal_metadata.voxel_x_nm))
sora_radius_px = int(round(crop_radius_um * 1000.0 / sora_metadata.voxel_x_nm))

for channel_index in CHANNEL_LABELS:
    pair = representative_pairs[channel_index]
    for row_index, (modality, nd2_path, radius_px, z, y, x) in enumerate([
        (
            "Conventional CSU-W1",
            NORMAL_BEAD_ND2,
            normal_radius_px,
            int(pair["normal_z"]),
            int(pair["normal_y"]),
            int(pair["normal_x"]),
        ),
        (
            "SoRa",
            SORA_BEAD_ND2,
            sora_radius_px,
            int(pair["sora_z"]),
            int(pair["sora_y"]),
            int(pair["sora_x"]),
        ),
    ]):
        axis = figure.add_subplot(image_grid[row_index, channel_index])
        image = normalized_local_projection(
            nd2_path,
            channel_index,
            z,
            y,
            x,
            radius_px,
            PRIMARY_PREPROCESSING_BY_MODALITY[modality],
        )
        axis.imshow(
            image,
            cmap="gray",
            vmin=0,
            vmax=1,
            extent=[-crop_radius_um, crop_radius_um, crop_radius_um, -crop_radius_um],
            interpolation="nearest",
        )
        if row_index == 0:
            axis.set_title(CHANNEL_LABELS[channel_index])
        if channel_index == 0:
            axis.set_ylabel(modality)
        axis.set_xticks([])
        axis.set_yticks([])
        for spine in axis.spines.values():
            spine.set_linewidth(0.6)
            spine.set_color("#555555")
        if row_index == 1 and channel_index == 2:
            bar_um = 0.5
            x_start = crop_radius_um - 0.08 - bar_um
            y_bar = crop_radius_um - 0.10
            axis.plot([x_start, x_start + bar_um], [y_bar, y_bar], color="white", linewidth=2.0, solid_capstyle="butt")
            axis.text(x_start + bar_um / 2, y_bar - 0.10, "500 nm", color="white", ha="center", va="bottom", fontsize=7)

first_image_axis = figure.axes[0]
first_image_axis.text(-0.20, 1.14, "a", transform=first_image_axis.transAxes, fontsize=12, fontweight="bold")

paired_axis = figure.add_subplot(outer[1, 0])
group_centers = np.arange(3) * 2.3
for channel_index, group_center in zip(CHANNEL_LABELS, group_centers):
    pairs = paired_beads.query("channel_index == @channel_index")
    x_normal, x_sora = group_center - 0.28, group_center + 0.28
    for pair in pairs.itertuples(index=False):
        paired_axis.plot(
            [x_normal, x_sora],
            [pair.normal_fwhm_xy_nm, pair.sora_fwhm_xy_nm],
            color="#B8B8B8",
            linewidth=0.65,
            alpha=0.70,
            zorder=1,
        )
    paired_axis.scatter(
        np.full(len(pairs), x_normal), pairs["normal_fwhm_xy_nm"],
        s=14, facecolor="white", edgecolor="#333333", linewidth=0.6, zorder=2,
    )
    paired_axis.scatter(
        np.full(len(pairs), x_sora), pairs["sora_fwhm_xy_nm"],
        s=14, facecolor=CHANNEL_COLORS[channel_index], edgecolor=CHANNEL_COLORS[channel_index], linewidth=0.6, zorder=3,
    )
    paired_axis.plot(
        [x_normal - 0.12, x_normal + 0.12],
        [pairs["normal_fwhm_xy_nm"].median()] * 2,
        color="black", linewidth=2.0, zorder=4,
    )
    paired_axis.plot(
        [x_sora - 0.12, x_sora + 0.12],
        [pairs["sora_fwhm_xy_nm"].median()] * 2,
        color="black", linewidth=2.0, zorder=4,
    )
    paired_axis.text(
        group_center, paired_axis.get_ylim()[0] if paired_axis.has_data() else 0,
        "", ha="center"
    )

paired_axis.set_xticks(group_centers)
paired_axis.set_xticklabels([CHANNEL_LABELS[index] for index in CHANNEL_LABELS])
paired_axis.set_xlabel("Detection channel")
paired_axis.set_ylabel("Observed lateral FWHM (nm)")
paired_axis.grid(axis="y", linestyle=":", color="#D0D0D0", linewidth=0.7)
paired_axis.spines[["top", "right"]].set_visible(False)
paired_axis.legend(
    handles=[
        Line2D([0], [0], marker="o", color="none", markerfacecolor="white", markeredgecolor="#333333", label="Conventional CSU-W1"),
        Line2D([0], [0], marker="o", color="none", markerfacecolor="#666666", markeredgecolor="#666666", label="SoRa"),
    ],
    loc="upper right", frameon=False,
)
paired_axis.text(-0.14, 1.05, "b", transform=paired_axis.transAxes, fontsize=12, fontweight="bold")

reduction_axis = figure.add_subplot(outer[1, 1])
y_positions = np.arange(len(paired_summary))
for y_position, row in paired_summary.iterrows():
    median = row["median_paired_reduction_percent"]
    low = row["bootstrap_95ci_low_percent"]
    high = row["bootstrap_95ci_high_percent"]
    reduction_axis.errorbar(
        median,
        y_position,
        xerr=[[median - low], [high - median]],
        fmt="o",
        markersize=5,
        color=CHANNEL_COLORS[int(row["channel_index"])],
        ecolor="#222222",
        elinewidth=1.0,
        capsize=3,
    )
    reduction_axis.text(
        high + 0.7,
        y_position,
        f"N={int(row['paired_beads'])}",
        va="center",
        fontsize=7.5,
    )
reduction_axis.axvline(0, color="black", linestyle="--", linewidth=0.8)
reduction_axis.set_yticks(y_positions)
reduction_axis.set_yticklabels(paired_summary["channel_label"])
reduction_axis.invert_yaxis()
reduction_axis.set_xlabel("Median paired reduction in FWHM (%)\nwithin-field bootstrap 95% CI")
reduction_axis.grid(axis="x", linestyle=":", color="#D0D0D0", linewidth=0.7)
reduction_axis.spines[["top", "right"]].set_visible(False)
reduction_axis.text(-0.28, 1.05, "c", transform=reduction_axis.transAxes, fontsize=12, fontweight="bold")

figure.text(
    0.02,
    0.015,
    "Primary analysis: raw conventional CSU-W1 and 3x3 XY median-filtered SoRa stacks; "
    "anisotropic 3D Gaussian fits. "
    "Images are independently normalized for display.",
    fontsize=7,
    color="#333333",
)
figure.subplots_adjust(left=0.08, right=0.97, top=0.95, bottom=0.11)

figure_base = FIGURE_DIR / "Supplementary_Fig_X_bead_width_Normal_vs_SoRa"
for suffix in ("pdf", "png", "svg"):
    figure.savefig(figure_base.with_suffix(f".{suffix}"), dpi=300, bbox_inches="tight")
plt.show()


## 6. Additional fit and registration QC


In [ ]:
qc_figure, axes = plt.subplots(1, 3, figsize=(10.5, 3.0), constrained_layout=True)

for channel_index in CHANNEL_LABELS:
    for modality, marker, facecolor in [
        ("Conventional CSU-W1", "o", "white"),
        ("SoRa", "o", CHANNEL_COLORS[channel_index]),
    ]:
        subset = primary_fits.query(
            "channel_index == @channel_index and modality == @modality"
        )
        x = np.full(len(subset), channel_index + (-0.12 if modality.startswith("Conventional") else 0.12))
        axes[0].scatter(
            x, subset["r_squared"], s=9, marker=marker,
            facecolor=facecolor, edgecolor="#444444", linewidth=0.35, alpha=0.65,
        )
axes[0].axhline(MIN_FIT_R_SQUARED, color="black", linestyle="--", linewidth=0.8)
axes[0].set_xticks(range(3), [CHANNEL_LABELS[i] for i in range(3)])
axes[0].set_ylabel("3D Gaussian fit R²")
axes[0].set_title("Accepted-fit quality")
axes[0].spines[["top", "right"]].set_visible(False)

for channel_index in CHANNEL_LABELS:
    pairs = paired_beads.query("channel_index == @channel_index")
    axes[1].scatter(
        np.full(len(pairs), channel_index), pairs["pair_distance_nm"],
        s=12, color=CHANNEL_COLORS[channel_index], alpha=0.75,
    )
axes[1].axhline(MAX_PAIR_DISTANCE_NM, color="black", linestyle="--", linewidth=0.8)
axes[1].set_xticks(range(3), [CHANNEL_LABELS[i] for i in range(3)])
axes[1].set_ylabel("Registration residual (nm)")
axes[1].set_title("Matched-bead registration")
axes[1].spines[["top", "right"]].set_visible(False)

for channel_index in CHANNEL_LABELS:
    pairs = paired_beads.query("channel_index == @channel_index")
    axes[2].scatter(
        np.full(len(pairs), channel_index), pairs["z_index_difference_sora_minus_normal"],
        s=12, color=CHANNEL_COLORS[channel_index], alpha=0.75,
    )
axes[2].axhline(0, color="black", linestyle="--", linewidth=0.8)
axes[2].set_xticks(range(3), [CHANNEL_LABELS[i] for i in range(3)])
axes[2].set_ylabel("SoRa - Normal bead Z index")
axes[2].set_title("Axial correspondence QC")
axes[2].spines[["top", "right"]].set_visible(False)

qc_base = FIGURE_DIR / "bead_fit_and_registration_QC"
qc_figure.savefig(qc_base.with_suffix(".pdf"), dpi=300, bbox_inches="tight")
qc_figure.savefig(qc_base.with_suffix(".png"), dpi=300, bbox_inches="tight")
plt.show()


## 7. Manuscript-ready text and machine-readable manifest


In [ ]:
summary_by_channel = paired_summary.set_index("channel_label")
result_parts = []
for label in ["640 nm", "515 nm", "445 nm"]:
    row = summary_by_channel.loc[label]
    result_parts.append(
        f"{label}: {row['normal_median_fwhm_xy_nm']:.0f} nm versus "
        f"{row['sora_median_fwhm_xy_nm']:.0f} nm "
        f"(median paired reduction {row['median_paired_reduction_percent']:.1f}%, "
        f"within-field bootstrap 95% CI {row['bootstrap_95ci_low_percent']:.1f}-{row['bootstrap_95ci_high_percent']:.1f}%; "
        f"N={int(row['paired_beads'])})"
    )

methods_text = (
    "Multispectral 200-nm fluorescent beads were imaged sequentially using "
    "conventional CSU-W1 and SoRa acquisition with the same Nikon Plan Apo lambda "
    "100x oil-immersion objective (NA 1.45) and 250-nm Z steps. The XY sampling "
    f"was {normal_metadata.voxel_x_nm:.1f} nm per pixel for conventional CSU-W1 "
    f"and {sora_metadata.voxel_x_nm:.1f} nm per pixel for SoRa. Candidate beads "
    "were detected from maximum-intensity projections after a 3x3 XY median "
    "filter. For the primary comparison, conventional CSU-W1 image widths were "
    "estimated from unfiltered 3D bead images, whereas SoRa image widths were "
    "estimated from 3x3 XY median-filtered 3D images to match the fixed-cell SoRa "
    "image-processing workflow. Beads were individually fit in three dimensions "
    "with an anisotropic "
    "Gaussian model. Fits with R-squared >= 0.80 and prespecified width bounds "
    "were retained. Corresponding beads in the overlapping field were matched "
    "using the known 2.8-fold sampling ratio, a robustly estimated XY translation, "
    "one-to-one assignment, and a maximum residual distance of 200 nm. Matched "
    "pairs were classified as aggregates and excluded when the lateral FWHM in "
    "both acquisitions exceeded the channel-specific upper Tukey fence (Q3 + "
    "1.5 x IQR); this criterion excluded two pairs in each detection channel. The observed "
    "lateral FWHM was calculated as the geometric mean of the fitted X and Y FWHM. "
    "Because the measurements include the finite 200-nm bead diameter and one "
    "technical acquisition was analyzed per modality, values are reported as "
    "observed bead image widths rather than absolute point-source resolution."
)

results_text = (
    "Measurements of matched 200-nm fluorescent beads acquired with the same "
    "objective showed narrower observed lateral image widths with SoRa than with "
    "conventional CSU-W1 acquisition across all three detection channels ("
    + "; ".join(result_parts)
    + "), confirming improved effective lateral imaging performance under the "
    "conditions used here (Supplementary Fig. X)."
)

legend_text = (
    "Supplementary Fig. X | Imaging performance of conventional CSU-W1 and SoRa "
    "measured using matched 200-nm fluorescent beads. (a) Representative matched "
    "beads in the 640-, 515-, and 445-nm detection channels. Conventional CSU-W1 "
    "images are unfiltered; SoRa images were processed with the 3x3 XY median "
    "filter used for the fixed-cell SoRa analysis. Images are independently normalized "
    "for display; scale bar, 500 nm. (b) Paired observed lateral FWHM values for "
    "beads identified in both acquisitions after excluding pairs that exceeded "
    "the channel-specific upper Tukey FWHM fence in both modalities. Lines connect "
    "the same bead; horizontal bars indicate medians. (c) Median within-bead percentage reduction in observed "
    "lateral FWHM with SoRa; error bars show within-field bootstrap 95% confidence intervals. "
    "N denotes matched beads from one technical acquisition per modality. The "
    "finite bead diameter is included in the reported image widths."
)

report_lines = [
    "# 200-nm bead Normal CSU-W1 versus SoRa comparison",
    "",
    "## Results sentence",
    "",
    results_text,
    "",
    "## Methods draft",
    "",
    methods_text,
    "",
    "## Supplementary figure legend draft",
    "",
    legend_text,
    "",
    "## Interpretation boundary",
    "",
    "- This is an optical technical-control acquisition, not a biological replicate series.",
    "- Individual beads quantify within-field dispersion and are not treated as independent experimental replicates.",
    "- Observed widths contain the finite 200-nm bead diameter and should not be labeled absolute PSF or absolute resolution.",
    "- The primary comparison uses raw conventional CSU-W1 fits and 3x3 XY median-filtered SoRa fits, avoiding a physically broader filter on the conventional images while preserving the fixed-cell SoRa preprocessing. All-raw sensitivity is exported separately; 445-nm SoRa raw fits do not satisfy the prespecified R2 threshold.",
    "- A channel-wise joint upper-FWHM outlier criterion excluded two spatially corresponding, visibly multi-lobed bead aggregates per channel; all pre-QC and excluded rows are exported.",
]
report_path = REPORT_DIR / "manuscript_methods_results_and_legend.md"
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest = {
    "analysis_date": "2026-08-10",
    "random_seed": SEED,
    "normal_bead_nd2": str(NORMAL_BEAD_ND2),
    "sora_bead_nd2": str(SORA_BEAD_ND2),
    "normal_nd2_size_bytes": NORMAL_BEAD_ND2.stat().st_size,
    "sora_nd2_size_bytes": SORA_BEAD_ND2.stat().st_size,
    "pipeline_script": str(PIPELINE_SCRIPT),
    "pipeline_sha256": sha256(PIPELINE_SCRIPT),
    "primary_preprocessing_by_modality": PRIMARY_PREPROCESSING_BY_MODALITY,
    "minimum_fit_r_squared": MIN_FIT_R_SQUARED,
    "maximum_pair_distance_nm": MAX_PAIR_DISTANCE_NM,
    "bootstrap_replicates": N_BOOTSTRAP,
    "bead_diameter_nm": 200.0,
    "paired_beads_by_channel": {
        row["channel_label"]: int(row["paired_beads"])
        for _, row in paired_summary.iterrows()
    },
    "aggregate_qc": {
        "criterion": (
            "Exclude a matched pair when both modality-specific lateral FWHM values "
            "exceed the channel-specific upper Tukey fence (Q3 + 1.5 x IQR)."
        ),
        "excluded_pairs_by_channel": {
            row["channel_label"]: int(row["excluded_aggregates"])
            for _, row in aggregate_qc_summary.iterrows()
        },
        "retained_pairs_by_channel": {
            row["channel_label"]: int(row["retained_after_qc"])
            for _, row in aggregate_qc_summary.iterrows()
        },
    },
    "outputs": {
        "figure_pdf": str(FIGURE_DIR / "Supplementary_Fig_X_bead_width_Normal_vs_SoRa.pdf"),
        "figure_png": str(FIGURE_DIR / "Supplementary_Fig_X_bead_width_Normal_vs_SoRa.png"),
        "figure_svg": str(FIGURE_DIR / "Supplementary_Fig_X_bead_width_Normal_vs_SoRa.svg"),
        "source_data": str(TABLE_DIR / "paired_bead_source_data.csv"),
        "aggregate_qc_source_data": str(
            TABLE_DIR / "paired_bead_aggregate_qc.csv"
        ),
        "excluded_aggregates": str(TABLE_DIR / "excluded_bead_aggregates.csv"),
        "summary": str(TABLE_DIR / "paired_bead_summary.csv"),
        "text_draft": str(report_path),
    },
}
(REPORT_DIR / "analysis_manifest.json").write_text(
    json.dumps(manifest, indent=2) + "\n", encoding="utf-8"
)

print(results_text)
print(f"\nReport: {report_path}")


## 8. Validation


In [ ]:
required_outputs = [
    FIGURE_DIR / "Supplementary_Fig_X_bead_width_Normal_vs_SoRa.pdf",
    FIGURE_DIR / "Supplementary_Fig_X_bead_width_Normal_vs_SoRa.png",
    FIGURE_DIR / "Supplementary_Fig_X_bead_width_Normal_vs_SoRa.svg",
    FIGURE_DIR / "bead_fit_and_registration_QC.pdf",
    TABLE_DIR / "paired_bead_source_data.csv",
    TABLE_DIR / "paired_bead_aggregate_qc.csv",
    TABLE_DIR / "excluded_bead_aggregates.csv",
    TABLE_DIR / "aggregate_qc_summary.csv",
    TABLE_DIR / "paired_bead_summary.csv",
    TABLE_DIR / "registration_summary.csv",
    REPORT_DIR / "manuscript_methods_results_and_legend.md",
    REPORT_DIR / "analysis_manifest.json",
]

validation = {
    "same_objective_name": normal_metadata.objective_name == sora_metadata.objective_name,
    "same_objective_na": math.isclose(normal_metadata.objective_na, sora_metadata.objective_na),
    "same_z_step": math.isclose(normal_metadata.voxel_z_nm, sora_metadata.voxel_z_nm),
    "normal_not_sora": not normal_metadata.sora,
    "sora_flag_present": sora_metadata.sora,
    "sampling_ratio_is_2_8": math.isclose(sampling_ratio, 2.8, rel_tol=1e-6),
    "primary_preprocessing_is_expected": bool(
        primary_fits.loc[
            primary_fits["modality"].eq("Conventional CSU-W1"), "preprocessing"
        ].eq("raw").all()
        and primary_fits.loc[
            primary_fits["modality"].eq("SoRa"), "preprocessing"
        ].eq("median3x3").all()
    ),
    "three_channels_present": set(paired_summary["channel_index"]) == {0, 1, 2},
    "two_aggregates_excluded_per_channel": bool(
        aggregate_qc_summary.set_index("channel_index")["excluded_aggregates"]
        .reindex(CHANNEL_LABELS)
        .eq(2)
        .all()
    ),
    "no_aggregate_flags_in_primary_data": not bool(
        paired_beads["excluded_as_aggregate"].any()
    ),
    "excluded_rows_are_joint_upper_outliers": bool(
        excluded_aggregates["normal_upper_fwhm_outlier"].all()
        and excluded_aggregates["sora_upper_fwhm_outlier"].all()
    ),
    "at_least_15_pairs_per_channel": bool((paired_summary["paired_beads"] >= 15).all()),
    "pair_residuals_within_threshold": bool(
        paired_beads["pair_distance_nm"].le(MAX_PAIR_DISTANCE_NM + 1e-9).all()
    ),
    "sora_narrower_in_all_channel_medians": bool(
        paired_summary["median_paired_reduction_percent"].gt(0).all()
    ),
    "all_required_outputs_exist": all(path.exists() and path.stat().st_size > 0 for path in required_outputs),
}
validation["all_checks_pass"] = all(validation.values())
validation_path = REPORT_DIR / "validation.json"
validation_path.write_text(json.dumps(validation, indent=2) + "\n", encoding="utf-8")

display(pd.DataFrame({"check": validation.keys(), "passed": validation.values()}))
if not validation["all_checks_pass"]:
    raise RuntimeError(f"Validation failed; inspect {validation_path}")
print(f"All validation checks passed. Output root: {OUTPUT_ROOT}")


## Completion criterion

The analysis is ready for manuscript use only when the final validation cell passes,
the representative and aggregate-QC images have been visually inspected, and the
finite-bead and single-acquisition limitations remain in the Methods/legend or
reviewer response.


In [ ]:
from matplotlib.lines import Line2D


LABEL_FONT_SIZE = 13      # 軸ラベル・画像行ラベル
OTHER_FONT_SIZE = 13    # 目盛り・凡例・タイトル・注記・パネル記号など

plt.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "font.size": OTHER_FONT_SIZE,
    "axes.titlesize": OTHER_FONT_SIZE,
    "axes.labelsize": OTHER_FONT_SIZE,
    "xtick.labelsize": OTHER_FONT_SIZE,
    "ytick.labelsize": OTHER_FONT_SIZE,
    "legend.fontsize": OTHER_FONT_SIZE,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


def normalized_local_projection(
    nd2_path: Path,
    channel_index: int,
    z: int,
    y: int,
    x: int,
    xy_radius_px: int,
    preprocessing: str,
    z_radius: int = 1,
) -> np.ndarray:
    with nd2.ND2File(nd2_path) as image:
        z_size = int(image.sizes["Z"])
        y_size = int(image.sizes["Y"])
        x_size = int(image.sizes["X"])
        z0, z1 = max(0, z - z_radius), min(z_size, z + z_radius + 1)
        y0, y1 = max(0, y - xy_radius_px), min(y_size, y + xy_radius_px + 1)
        x0, x1 = max(0, x - xy_radius_px), min(x_size, x + xy_radius_px + 1)
        local_stack = np.asarray(
            image.to_dask()[z0:z1, channel_index, y0:y1, x0:x1].compute()
        )
    if preprocessing == "median3x3":
        local_stack = median_filter(local_stack, size=(1, 3, 3))
    elif preprocessing != "raw":
        raise ValueError(f"Unsupported preprocessing: {preprocessing}")
    crop = local_stack.max(axis=0).astype(np.float32)
    background = float(np.percentile(crop, 10.0))
    crop = np.clip(crop - background, 0.0, None)
    scale = float(np.percentile(crop, 99.8))
    return np.clip(crop / scale, 0.0, 1.0) if scale > 0 else crop


# Select the retained pair closest to the channel-wise bivariate median of the
# conventional and SoRa absolute FWHM values. Fit quality and registration residual
# provide deterministic tie-breakers.
representative_pairs = {}
for channel_index, pairs in paired_beads.groupby("channel_index", sort=True):
    normal_median = float(pairs["normal_fwhm_xy_nm"].median())
    sora_median = float(pairs["sora_fwhm_xy_nm"].median())
    candidates = pairs.assign(
        distance_to_bivariate_median=np.hypot(
            np.log(pairs["normal_fwhm_xy_nm"] / normal_median),
            np.log(pairs["sora_fwhm_xy_nm"] / sora_median),
        ),
        minimum_r_squared=pairs[
            ["normal_r_squared", "sora_r_squared"]
        ].min(axis=1),
    ).sort_values(
        ["distance_to_bivariate_median", "minimum_r_squared", "pair_distance_nm"],
        ascending=[True, False, True],
    )
    representative_pairs[int(channel_index)] = candidates.iloc[0]


channel_indices = list(CHANNEL_LABELS)
figure = plt.figure(figsize=(13.6, 4.3), constrained_layout=False)
outer = figure.add_gridspec(
    1,
    2,
    width_ratios=[1.55, 1.0],
    wspace=0.28,
)
image_grid = outer[0, 0].subgridspec(
    2,
    len(channel_indices),
    hspace=0.12,
    wspace=0.12,
)

crop_radius_um = 0.75
normal_radius_px = int(
    round(crop_radius_um * 1000.0 / normal_metadata.voxel_x_nm)
)
sora_radius_px = int(
    round(crop_radius_um * 1000.0 / sora_metadata.voxel_x_nm)
)

first_image_axis = None
for column_index, channel_index in enumerate(channel_indices):
    pair = representative_pairs[channel_index]
    modalities = [
        (
            "Conventional\n CSU-W1",
            PRIMARY_PREPROCESSING_BY_MODALITY["Conventional CSU-W1"],
            NORMAL_BEAD_ND2,
            normal_radius_px,
            int(pair["normal_z"]),
            int(pair["normal_y"]),
            int(pair["normal_x"]),
        ),
        (
            "SoRa",
            PRIMARY_PREPROCESSING_BY_MODALITY["SoRa"],
            SORA_BEAD_ND2,
            sora_radius_px,
            int(pair["sora_z"]),
            int(pair["sora_y"]),
            int(pair["sora_x"]),
        ),
    ]
    for row_index, (
        modality,
        preprocessing,
        nd2_path,
        radius_px,
        z,
        y,
        x,
    ) in enumerate(modalities):
        axis = figure.add_subplot(image_grid[row_index, column_index])
        if first_image_axis is None:
            first_image_axis = axis
        image = normalized_local_projection(
            nd2_path,
            channel_index,
            z,
            y,
            x,
            radius_px,
            preprocessing,
        )
        axis.imshow(
            image,
            cmap="gray",
            vmin=0,
            vmax=1,
            extent=[
                -crop_radius_um,
                crop_radius_um,
                crop_radius_um,
                -crop_radius_um,
            ],
            interpolation="nearest",
        )
        if row_index == 0:
            axis.set_title(CHANNEL_LABELS[channel_index])
        if column_index == 0:
            axis.set_ylabel(modality)
        axis.set_xticks([])
        axis.set_yticks([])
        for spine in axis.spines.values():
            spine.set_linewidth(0.6)
            spine.set_color("#555555")

        if row_index == 1 and column_index == len(channel_indices) - 1:
            bar_um = 0.5
            x_start = crop_radius_um - 0.08 - bar_um
            y_bar = crop_radius_um - 0.10
            axis.plot(
                [x_start, x_start + bar_um],
                [y_bar, y_bar],
                color="white",
                linewidth=2.0,
                solid_capstyle="butt",
            )
            axis.text(
                x_start + bar_um / 2,
                y_bar - 0.10,
                "500 nm",
                color="white",
                ha="center",
                va="bottom",
                fontsize=7,
            )

first_image_axis.text(
    -0.20,
    1.16,
    "",
    transform=first_image_axis.transAxes,
    fontsize=12,
    fontweight="bold",
)


paired_axis = figure.add_subplot(outer[0, 1])
group_centers = np.arange(len(channel_indices), dtype=float) * 2.3
for channel_index, group_center in zip(channel_indices, group_centers):
    pairs = paired_beads.query("channel_index == @channel_index")
    x_normal = group_center - 0.28
    x_sora = group_center + 0.28

    for pair in pairs.itertuples(index=False):
        paired_axis.plot(
            [x_normal, x_sora],
            [pair.normal_fwhm_xy_nm, pair.sora_fwhm_xy_nm],
            color="#B8B8B8",
            linewidth=0.65,
            alpha=0.70,
            zorder=1,
        )

    paired_axis.scatter(
        np.full(len(pairs), x_normal),
        pairs["normal_fwhm_xy_nm"],
        s=14,
        facecolor="white",
        edgecolor="#333333",
        linewidth=0.6,
        zorder=2,
    )
    paired_axis.scatter(
        np.full(len(pairs), x_sora),
        pairs["sora_fwhm_xy_nm"],
        s=14,
        facecolor=CHANNEL_COLORS[channel_index],
        edgecolor=CHANNEL_COLORS[channel_index],
        linewidth=0.6,
        zorder=3,
    )

    normal_median = float(pairs["normal_fwhm_xy_nm"].median())
    sora_median = float(pairs["sora_fwhm_xy_nm"].median())
    paired_axis.plot(
        [x_normal - 0.12, x_normal + 0.12],
        [normal_median, normal_median],
        color="black",
        linewidth=2.0,
        zorder=4,
    )
    paired_axis.plot(
        [x_sora - 0.12, x_sora + 0.12],
        [sora_median, sora_median],
        color="black",
        linewidth=2.0,
        zorder=4,
    )

paired_axis.set_xticks(group_centers)
paired_axis.set_xticklabels(
    [CHANNEL_LABELS[index] for index in channel_indices]
)
paired_axis.set_xlabel("Detection channel")
paired_axis.set_ylabel("Observed lateral FWHM (nm)")
paired_axis.grid(
    axis="y",
    linestyle=":",
    color="#D0D0D0",
    linewidth=0.7,
)
paired_axis.spines[["top", "right"]].set_visible(False)
paired_axis.legend(
    handles=[
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor="white",
            markeredgecolor="#333333",
            label="CSU-W1",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor="#666666",
            markeredgecolor="#666666",
            label="SoRa",
        ),
    ],
    loc="upper right",
    frameon=False,
)
paired_axis.text(
    -0.16,
    1.05,
    "",
    transform=paired_axis.transAxes,
    fontsize=12,
    fontweight="bold",
)

paired_axis.set_ylim(top=300)

# figure.text(
#     0.02,
#     0.018,
#     "Primary analysis: raw conventional CSU-W1 and 3x3 XY median-filtered SoRa stacks; "
#     "anisotropic 3D Gaussian fits. "
#     "Images are independently normalized for display.",
#     fontsize=7,
#     color="#333333",
# )
figure.subplots_adjust(
    left=0.07,
    right=0.98,
    top=0.90,
    bottom=0.18,
)

figure_base = (
    FIGURE_DIR
    / "Supplementary_Fig_X_bead_width_Normal_vs_SoRa_panels_d_e"
)
for suffix in ("pdf", "png", "svg"):
    figure.savefig(
        figure_base.with_suffix(f".{suffix}"),
        dpi=300,
        bbox_inches="tight",
    )
plt.show()